# Fairness Evaluation

- This notebook will help me to analyse the correlation (if any) between the model I have created and the protected characteristics I have specifically excluded in building the model.
- I have chosen a group of 3 attributes, `age_years`, `sex` and `region`.
- I will be using False Positive Rates, False Negative Rates and decline rates in order to evaluate this.
- Please see decision journal entry 012 for further details on my menthodical justification.

In [ ]:
import pandas as pd

df = pd.read_csv("../data/creditsense_applications.csv")

print("Original dataset shape:", df.shape)

In [ ]:
df_model = df.dropna(
    subset=["savings_balance_gbp", "years_at_current_address"]
).copy()

print("Modelling dataset shape:", df_model.shape)
print("Rows excluded:", len(df) - len(df_model))

In [ ]:
excluded_features = [
    "application_id",
    "default_within_24m",
    "age_years",
    "sex",
    "marital_status",
    "residency_status",
    "num_dependents",
    "region",
    "loan_purpose",
    "application_channel",
    "previous_application_declined"
]

In [ ]:
X = df_model.drop(columns=excluded_features)

y = df_model["default_within_24m"]

print("predictive features:", X.shape[1])
print("Target:", y.name)
print("\nSelected features:")
print(X.columns.tolist())

In [ ]:
nominal_features = [
    "employment_status",
    "housing_status",
    "credit_history",
    "checking_account_status"
]

binary_features = [
    "has_savings_account",
    "has_guarantor"
]

numerical_features = [
    col for col in X.columns
    if col not in nominal_features + binary_features
]

In [ ]:
print("Nominal:", nominal_features)
print("Binary:", binary_features)
print("Numerical:", numerical_features)

print("\nTotal features:",
      len(nominal_features)
      + len(binary_features)
      + len(numerical_features))

In [ ]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

In [ ]:
print("Training records:", X_train.shape[0])
print("Testing records:", X_test.shape[0])

print("\nTraining default rate:")
print(y_train.value_counts(normalize=True))

print("\nTesting default rate:")
print(y_test.value_counts(normalize=True))

In [ ]:
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import (
    StandardScaler,
    OneHotEncoder,
    OrdinalEncoder
)

In [ ]:
preprocessor = ColumnTransformer(
    transformers=[
        ("num", StandardScaler(), numerical_features),
        ("nom", OneHotEncoder(handle_unknown="ignore"), nominal_features),
        ("bin", OrdinalEncoder(
            categories=[["No", "Yes"], ["No", "Yes"]]
        ), binary_features)
    ]
)

In [ ]:
from sklearn.pipeline import Pipeline

baseline_model = Pipeline(
    steps =[
        ("preprocessor", preprocessor),
        ("classifier", LogisticRegression(
            max_iter=1000,
            random_state=42
        ))
    ]
)

In [ ]:
from sklearn.linear_model import LogisticRegression

baseline_model = Pipeline(
    steps =[
        ("preprocessor", preprocessor),
        ("classifier", LogisticRegression(
            max_iter=1000,
            random_state=42
        ))
    ]
)

In [ ]:
from sklearn.model_selection import StratifiedKFold, cross_val_score

cv = StratifiedKFold(
    n_splits=5,
    shuffle=True,
    random_state=42
)

In [ ]:
cv_scores = cross_val_score(
    baseline_model,
    X_train,
    y_train,
    cv=cv,
    scoring="roc_auc"
)

print("ROC_AUC per fold:", cv_scores)
print("Mean ROAC_AUC:", cv_scores.mean())
print("Standard deviation:", cv_scores.std())

In [ ]:
from sklearn.model_selection import cross_val_predict

lr_cv_predictions = cross_val_predict(
    baseline_model,
    X_train,
    y_train,
    cv=cv,
    method="predict"
)

In [ ]:
from sklearn.metrics import confusion_matrix

lr_cm = confusion_matrix(y_train, lr_cv_predictions)

print("Logistic Regression:")
print(lr_cm)

In [ ]:
lr_cv_probabilities = cross_val_predict(
    baseline_model,
    X_train,
    y_train,
    cv=cv,
    method="predict_proba"
)

All of the above cells have been directly copied from 03_model_comparison.ipynb to keep consistency.

In [ ]:
lr_default_probabilities = lr_cv_probabilities[:, 1]

In [ ]:
lr_default_probabilities[:5]

Filtered the probabilities down to just the probability of default and then showed the first five to ensure it has worked.

In [ ]:
threshold = 0.48

lr_predictions = (lr_default_probabilities >= threshold).astype(int)

In [ ]:
lr_predictions[:5]

Confirmation that the threshold is responsing correctly by showing the first five results.

In [ ]:
df_model.columns.tolist()

Confirming the correct model name to use for the fairness evaluation.

In [ ]:
df_model.loc[X_train.index, X_train.columns].equals(X_train)

Confirming that `df_model` corresponds to the training data.

In [ ]:
fairness_df = df_model.loc[X_train.index, ["age_years", "sex", "region"]].copy()

In [ ]:
fairness_df.head()

Ensuring I have been able to create a dataframe aligned with the information requires. This explicitly protects these characteristics from being introduced to the actual predictive model.

In [ ]:
fairness_df.index.equals(y_train.index) and fairness_df.index.is_unique and len(fairness_df) == len(lr_default_probabilities) == len(lr_predictions)

Confirming index alignment and row counts.

In [ ]:
fairness_df["actual_default"] = y_train
fairness_df["predicted_default"] = lr_predictions
fairness_df["default_probability"] = lr_default_probabilities

Assigning each column to the right data.

In [ ]:
fairness_df.head()

Confirming the creation of the datframe has been successful.

In [ ]:
fairness_df.isna().sum()

Checking there are no missing values in the dataframe.

In [ ]:
fairness_df["sex"].value_counts()

Checking how many of each gender occur in our dataframe. Only 93 less women than men. No substantial representation issues.

In [ ]:
pd.crosstab(
    fairness_df["sex"],
    fairness_df["actual_default"]
)

Examining the relationship between sex and observed default outcomes before the model gets involved. Raw counts show women have more applicants that default.

In [ ]:
pd.crosstab(
    fairness_df["sex"],
    fairness_df["actual_default"],
    normalize="index"
)

Converting the data into proportions to make them more comparable. Around 3.76 percentage points difference in default rate between men and women looking at just the raw data. Note: We have not linked this to financial circumstances.

In [ ]:
pd.crosstab(
    fairness_df["sex"],
    fairness_df["predicted_default"],
    normalize="index"
)

Looking at the models lending decision to compare to the actual outcomes. There is an increase of 0.89 percentage points of difference between mens and womens default rates.

In [ ]:
female_df = fairness_df[fairness_df["sex"] == "Female"]

In [ ]:
female_df

Filtering just the female applicants out of the dataframe.

In [ ]:
female_cm = confusion_matrix(
    female_df["actual_default"],
    female_df["predicted_default"],
    labels=[0, 1]
)

female_cm

Validating the previous cross-tab.

In [ ]:
tn, fp, fn, tp = female_cm.ravel()

In [ ]:
female_fpr = fp / (fp + tn)
female_fnr = fn / (fn + tp)

print("FPR", female_fpr)
print("FNR", female_fnr)

Generating the false positive rate and false negative rate for evaluation.

In [ ]:
import numpy as np

def calculate_fairness_metric(group_df):
    cm = confusion_matrix(
        group_df["actual_default"],
        group_df["predicted_default"],
        labels=[0, 1]
    )

    tn, fp, fn, tp = cm.ravel()
    fpr = fp / (fp + tn) if (fp + tn) > 0 else np.nan
    fnr = fn / (fn + tp) if (fn + tp) > 0 else np.nan
    group_size = tn + fn + tp + fp
    actual_default_rate = (fn + tp) / group_size if group_size > 0 else np.nan
    decline_rate = (fp + tp) / group_size if group_size > 0 else np.nan
    return {
        "group_size": group_size,
        "FPR": fpr,
        "FNR": fnr,
        "actual_default_rate": actual_default_rate,
        "decline_rate": decline_rate
    }

Creating a repeatable python function in order to simplify the process for the other variables.

In [ ]:
calculate_fairness_metric(female_df)

Confirming our repeatable python function returns the known values from earlier before using for other variables.

In [ ]:
for group_name, group_df in fairness_df.groupby("sex"):
    print(group_name)
    print(calculate_fairness_metric(group_df))

Creating a dictionary of the seperated genders for use in a dataframe.

In [ ]:
sex_results = []

for group_name, group_df in fairness_df.groupby("sex"):
    metrics = calculate_fairness_metric(group_df)
    metrics["sex"] = group_name
    sex_results.append(metrics)

In [ ]:
sex_fairness_df = pd.DataFrame(sex_results)

sex_fairness_df

Showing the fully functioning dataframe. This has highlighted the disparity in female to male rates. Women are 3.44 percentage points more exposed to incorrect declines. Men are 2.37 percentage points more exposed to incorrect approvals. Male applicants are less likely to be declined overall and less likely to be declined incorrectly. This does mean men are more likely to be incorrectly approved and therefore potetially exposed to financial hardship. Women are approximately 56% more likely to be incorrectly declined than men in the same position making it harder for them to get potential financial support. This does not imply causation. This is all exploration, not used on held-out test-data.

In [ ]:
fairness_df["region"].value_counts()

Notably less applicants in Northern Ireland.

In [ ]:
pd.crosstab(
    fairness_df["region"],
    fairness_df["actual_default"],
    normalize="index"
    )

Northern Ireland has only 25 actual defaults so any false negative would change the FNR by 4 percentage points, yet it also has the highest observed default rate. South West is showing the lowest with a difference of 7.40 percentage points.

In [ ]:
pd.crosstab(
    fairness_df["region"],
    fairness_df["predicted_default"],
    normalize="index"
)